# 第81课 · 推荐系统竟是一道算术题——嵌入向量（embedding）均值 + k-NN，亲手写出懂你的歌单

**目标**：实现完整推荐流程——给定用户喜欢的 N 首歌，计算用户 profile embedding，找最近邻，过滤已听歌曲，输出推荐列表。

🔗 **Aurora 连接**：本节是 Aurora Music Intelligence Engine 的核心交付物，推荐逻辑对应 `src/aurora/music/similarity.py`（`find_similar`）；歌曲特征提取对应 `src/aurora/music/features.py`；歌曲嵌入对应 `src/aurora/music/embed.py`（L79 的 `MusicEncoder`，已存在）。（`recommend.py` / `sim.py` 为规划中桩文件，尚未创建。）

← **上一课**　[L80 · 向量相似度检索](L80_similarity.ipynb)

> 上节课学习了 **向量相似度检索**：余弦相似度 vs 点积 vs L2，纯 NumPy k-NN 实现。  
> 本课将探讨 **音乐推荐系统**。

## 学习目标

完成本节后，你应该能够：

1. **实现 mean-embedding user profile**：从用户历史 id 列表计算归一化平均嵌入向量。
2. **处理冷启动问题**：当历史为空时，回退到全局均值嵌入作为中性出发点。
3. **实现过滤逻辑**：在推荐结果中正确排除历史已听歌曲。
4. **评估 embedding 质量对 Precision@K 的影响**：通过控制噪声系数实验，理解嵌入质量与推荐精度之间的关系。

> **Aurora 连接**：`aurora.music.similarity`（相似度检索）、`aurora.music.features`（特征提取）

## 本课剧情：Netflix 的惊天秘密——推荐系统居然就是一道算术题

Netflix 有 2 亿用户，每人每天看不同的内容。它的推荐算法听起来神秘，但核心思路极其简单：

**你喜欢的电影/歌曲的 embedding 向量的平均值，就是"你这个人"的向量。**

```
用户历史：《哈利波特》embedding  +  《魔戒》embedding  +  《纳尼亚》embedding
         ──────────────────────────────────────────────────────────
                           ÷ 3（平均）
                               ↓
                      user_embedding（奇幻爱好者向量）
                               ↓
                      找最近的 top-10 个歌曲/电影
                               ↓
                      推荐列表（过滤掉已看过的）
```

**三个设计问题**：

| 问题 | 解法 |
|---|---|
| 用户历史为空（新用户）| 冷启动：用全库均值，或热门榜单 |
| 历史歌曲出现在推荐中 | 过滤：从 top-k 候选中删除已听歌曲 |
| 用户口味漂移（时间变化）| 加权平均：近期历史权重更高 |

**Precision@k**：评估推荐质量的指标。若用 ground-truth "用户后续会喜欢的歌" 来验证，Precision@5 = 推荐5首中有几首真的被喜欢 / 5。

本节任务：实现 `recommend()`，把历史→平均embedding→过滤→top-k 串联成一个完整推荐函数。

In [ ]:
import numpy as np

# mu3_sim 若已实现，取消注释:
# from aurora.music.similarity import find_similar  # 正确命名空间（旧: month04.mu3_sim 已废弃）

# 本节自包含版本：直接用 numpy 实现相似度检索
np.random.seed(42)


## 1. 用户 Profile = mean(liked song embeddings)

给定用户喜欢的歌曲 id 列表 `history = [i₀, i₁, ..., iₙ]`，从全量 embedding 矩阵 `all_embs`（形状 `[N, d]`）取出对应行，求平均：

```
user_emb = all_embs[history].mean(axis=0)   # shape: (d,)
```

然后做 L2 归一化，把模长变成 1，这样就能用点积代替余弦相似度（cosine similarity）：

```
user_emb = user_emb / (np.linalg.norm(user_emb) + 1e-8)
```

这等价于协同过滤中"用户对喜欢的物品取平均隐向量"的做法（Item-based CF 的对偶视角）。embedding 质量越高（同类歌曲聚集越紧），平均向量就越能代表用户真实口味。


In [ ]:
# 演示：3首歌的 embedding 平均
N, d = 100, 8
all_embs = np.random.randn(N, d)
# 归一化所有 embedding（模拟已归一化的 song embeddings）
all_embs = all_embs / (np.linalg.norm(all_embs, axis=1, keepdims=True) + 1e-8)

history = [0, 5, 12]
user_emb_raw = all_embs[history].mean(axis=0)
user_emb = user_emb_raw / (np.linalg.norm(user_emb_raw) + 1e-8)
print(f"user_emb shape : {user_emb.shape}")
print(f"user_emb norm  : {np.linalg.norm(user_emb):.4f}  (应 ≈ 1.0)")
print(f"user_emb[:4]   : {user_emb[:4].round(4)}")


## 2. 冷启动（cold start）问题：新用户没有历史怎么办？

当 `history = []` 时，无法计算平均 embedding。两种常见应对策略：

**策略 A — 流行度排名**：预先计算全库每首歌的播放次数，冷启动时返回全局 Top-K。公式上等价于用全量 embedding 的全局均值作为 user_emb（或直接按 popularity score 排序）：

```
# 全局平均（几何中心）作为冷启动 user_emb
cold_emb = all_embs.mean(axis=0)
cold_emb = cold_emb / (np.linalg.norm(cold_emb) + 1e-8)
```

**策略 B — 随机多样化**：随机采样若干歌曲作为初始推荐，降低马太效应（Matthew effect），同时收集第一批交互信号。

策略 A 可解释性强；策略 B 探索性强。工业界通常先 A 再 B 混合（`ε-greedy`）。


In [ ]:
# 演示：冷启动 — 全局均值 vs 随机采样
cold_emb_global = all_embs.mean(axis=0)
cold_emb_global /= np.linalg.norm(cold_emb_global) + 1e-8

scores_cold = all_embs @ cold_emb_global
top_cold = np.argsort(scores_cold)[::-1][:5]
print(f"冷启动 Top-5（全局均值策略）: {top_cold.tolist()}")

# 随机多样化策略
top_random = np.random.choice(N, size=5, replace=False)
print(f"冷启动 Top-5（随机策略）    : {top_random.tolist()}")


## 3. 过滤已听：推荐结果去掉已喜欢的歌

排序后的候选列表中，历史歌曲得分必然偏高（它们定义了 user_emb），直接推荐会造成重复——用户已经知道这首歌，推给他毫无价值。

过滤方法：把 `history` 转成集合，在取 Top-K 时跳过集合内的 id：

```
history_set = set(user_history_ids)
ranked = np.argsort(scores)[::-1]
recs = [i for i in ranked if i not in history_set][:top_k]
```

时间复杂度：`O(N)` 排序 + `O(K)` 过滤，对百万量级需换用 FAISS + 后处理。Aurora 当前规模（万级歌曲）直接用 numpy 排序即可。


In [ ]:
# 演示：过滤前 vs 过滤后
scores_demo = all_embs @ user_emb
ranked_all = np.argsort(scores_demo)[::-1][:8]
print(f"过滤前 Top-8: {ranked_all.tolist()}")
print(f"history ids : {history}")

history_set = set(history)
ranked_filtered = [i for i in np.argsort(scores_demo)[::-1] if i not in history_set][:8]
print(f"过滤后 Top-8: {ranked_filtered}")
print(f"（历史歌曲 {[i for i in ranked_all if i in history_set]} 已被移除）")


## 4. ✏️ 实现 `recommend(user_history_ids, all_embs, top_k=10)`

**签名**：
```python
def recommend(
    user_history_ids: list[int],   # 用户历史歌曲 id 列表
    all_embs: np.ndarray,          # (N, d) 所有歌曲的 embedding（已 L2 归一化）
    top_k: int = 10
) -> np.ndarray:
    # 返回: (top_k,) int，推荐歌曲 id，按相似度降序
```

**4步实现路线**：

| 步骤 | 操作 | 边界情况 |
|---|---|---|
| 1 | 计算 user_emb | `all_embs[history_ids].mean(axis=0)`；若 history 为空→用全库均值 |
| 2 | L2 归一化 user_emb | `/ np.linalg.norm(user_emb)` |
| 3 | 计算所有歌曲的余弦相似度 | `scores = all_embs @ user_emb` |
| 4 | 过滤历史，取 top-k | 先将 `scores[history_ids] = -inf`，再 `argsort` 取后 top_k |

**验收标准**：
- 空历史 → 返回 top_k 个 id（不报错）
- 推荐结果不含 history 中的歌曲
- 结果按相似度降序排列，shape=(top_k,)

In [ ]:
def recommend(
    user_history_ids: list[int],
    all_embs: "np.ndarray",
    top_k: int = 10,
) -> "np.ndarray":
    """
    user_history_ids : list[int]   用户喜欢的歌曲 id
    all_embs         : np.ndarray  shape (N, d)，L2 归一化的歌曲 embedding
    top_k            : int         返回条数
    返回             : np.ndarray  shape (top_k,) int，推荐歌曲 id，按相似度降序
    """
    # ✏️ TODO 步骤1：处理冷启动（history 为空）→ 用全局均值
    # ✏️ TODO 步骤2：计算 user_emb = all_embs[history_ids].mean(axis=0)，L2 归一化
    # ✏️ TODO 步骤3：scores = all_embs @ user_emb
    # ✏️ TODO 步骤4：scores[history_ids] = -inf，argsort 取后 top_k（降序）
    raise NotImplementedError("TODO: 基于余弦相似度返回 top-k 推荐列表")

In [ ]:
# ── 自动检查 ──────────────────────────────────────────────────────────────────
np.random.seed(0)
N_test, d_test = 50, 8

# 构造有聚类结构的 embedding：前10首歌属于"爵士簇"
jazz_center = np.array([1.0, 0.5, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0])
embs_test = np.random.randn(N_test, d_test) * 0.3
embs_test[:10] += jazz_center          # 爵士簇：id 0-9
embs_test = embs_test / (np.linalg.norm(embs_test, axis=1, keepdims=True) + 1e-8)

# 用户喜欢5首爵士（id 0-4）
history_test = [0, 1, 2, 3, 4]
try:
    recs = recommend(history_test, embs_test, top_k=5)
except (NotImplementedError, TypeError):
    recs = None

if recs is None:
    print("⬜ 未实现，跳过验证")
else:
    assert len(recs) == 5, f"期望5条推荐，得到 {len(recs)}"
    assert all(isinstance(r, (int, np.integer)) for r in recs), "推荐 id 应为 int 或 np.integer（np.argsort 返回 np.int64）"
    assert not any(r in set(history_test) for r in recs), "推荐中不应含历史歌曲"

    # 推荐结果应以爵士簇（id 5-9）为主
    jazz_hits = sum(1 for r in recs if r < 10)
    assert jazz_hits >= 3, f"期望推荐 ≥3 首爵士歌曲，实际 {jazz_hits}（embedding 聚类质量或实现有误）"

    print(f"✅ recommend() 通过：推荐列表 = {recs}，爵士命中 {jazz_hits}/5")

    # 冷启动检查
    recs_cold = recommend([], embs_test, top_k=3)
    assert recs_cold is not None and len(recs_cold) == 3, "冷启动应返回3条"
    print(f"✅ 冷启动通过：{recs_cold}")

## 5. 参数实验：模拟 10 个用户，画 Precision@5 分布

**实验设计**：
- `n_users = 10`，每个用户随机选 `history_len ∈ [3, 10]` 首歌作为历史
- `top_k = 5`，推荐后统计 Precision@5：推荐中属于"同风格簇"的比例
- 控制变量：改变 **embedding 噪声系数** `noise_std`（0.1 → 0.5 → 1.0），  观察 Precision@5 随 embedding 质量下降的变化趋势

**预期现象**（`n_genres = 5`，随机基线 = 1/5 = 0.2）：
- `noise_std = 0.1`（高质量 embedding，簇紧密）：Precision@5 接近满分，本实验 mean ≈ 1.00
- `noise_std = 0.5`（中等质量）：开始出现错配，本实验 mean ≈ 0.84
- `noise_std = 1.0`（低质量 embedding，簇散乱）：明显下滑，但因 5 个类中心仍是彼此区分的单位向量，尚未跌到随机基线 0.2，本实验 mean ≈ 0.46
- 历史越长（history_len 越大），平均 embedding 越稳定，Precision@5 方差越小


In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

def run_experiment(noise_std, n_users=10, n_songs=200, d=16, n_genres=5, top_k=5):
    """生成多风格 embedding，模拟用户，计算 Precision@5"""
    np.random.seed(42)
    songs_per_genre = n_songs // n_genres  # 每类歌曲数

    # 构造 embedding：每类歌曲围绕一个随机中心
    centers = np.random.randn(n_genres, d)
    centers /= np.linalg.norm(centers, axis=1, keepdims=True) + 1e-8
    embs = np.vstack([
        centers[g] + np.random.randn(songs_per_genre, d) * noise_std
        for g in range(n_genres)
    ])
    embs /= np.linalg.norm(embs, axis=1, keepdims=True) + 1e-8

    # 每首歌的真实类别标签
    labels = np.repeat(np.arange(n_genres), songs_per_genre)

    precisions = []
    for u in range(n_users):
        # 随机选一个主类别作为该用户偏好
        fav_genre = np.random.randint(n_genres)
        genre_ids = np.where(labels == fav_genre)[0].tolist()

        hist_len = np.random.randint(3, min(11, len(genre_ids)))
        history = np.random.choice(genre_ids, size=hist_len, replace=False).tolist()

        try:
            recs = recommend(history, embs, top_k=top_k)
        except (NotImplementedError, TypeError):
            recs = None
        if recs is None:
            precisions.append(0.0)
            continue
        hits = sum(1 for r in recs if labels[r] == fav_genre)
        precisions.append(hits / top_k)

    return precisions

noise_levels = [0.1, 0.5, 1.0]
labels_plot  = ["高质量 (noise=0.1)", "中等 (noise=0.5)", "低质量 (noise=1.0)"]
colors       = ["#2ecc71", "#f39c12", "#e74c3c"]

fig, axes = plt.subplots(1, 3, figsize=(12, 4), sharey=True)
fig.suptitle("Precision@5 分布 vs Embedding 质量（10 用户模拟）", fontsize=13)

for ax, noise, label, color in zip(axes, noise_levels, labels_plot, colors):
    precs = run_experiment(noise_std=noise)
    ax.bar(range(len(precs)), precs, color=color, alpha=0.8)
    ax.axhline(np.mean(precs), color="black", linestyle="--", linewidth=1.2,
               label=f"mean={np.mean(precs):.2f}")
    ax.set_title(label, fontsize=11)
    ax.set_xlabel("用户编号")
    ax.set_ylim(0, 1.05)
    ax.legend(fontsize=9)

axes[0].set_ylabel("Precision@5")
plt.tight_layout()
plt.savefig("mu4_precision_experiment.png", dpi=100, bbox_inches="tight")
plt.show()
print("✅ 实验图已保存至 mu4_precision_experiment.png")
print()

# 打印均值汇总
for noise, label in zip(noise_levels, labels_plot):
    precs = run_experiment(noise_std=noise)
    print(f"{label}: mean Precision@5 = {np.mean(precs):.3f}  std = {np.std(precs):.3f}")


## 本课收束

本节实现了 `recommend(user_history_ids, all_embs, top_k)`，输出为过滤已听歌曲后的推荐 id 列表（`list[int]`，长度 `top_k`）。该函数的实际对应模块为 `src/aurora/music/similarity.py`（相似度检索）、`src/aurora/music/features.py`（特征提取）和 `src/aurora/music/embed.py`（嵌入编码器，已存在），`recommend.py` 为规划中桩文件（可替换 numpy 排序为 FAISS）。下一节（L82）将通过 Chroma 热力图和 Embedding t-SNE 散点图，直观验证音乐嵌入的聚类效果。


## ✏️ 白板挑战：推荐系统手推（目标 10 分钟）

盖上屏幕，纸上推导：

**问 1**：mean-embedding user profile 的几何含义是什么？  
（历史歌曲向量的"重心"（centroid）；离重心最近的歌曲在语义上最接近用户综合口味）

**问 2**：若用户历史有 3 首歌，embedding=[1,0], [0,1], [1,1]，user_emb=？L2归一化后=？  
（平均=[2/3, 2/3]；模长=√(4/9+4/9)=2√2/3；归一化=[1/√2, 1/√2]≈[0.707, 0.707]）

**问 3**：为什么推荐时要过滤已听歌曲？从数学上解释。  
（历史歌曲的embedding直接参与了user_emb的计算，与user_emb余弦相似度极高（接近1），若不过滤必然霸占top位，推荐全是已听歌曲）

**问 4**：冷启动问题（history=[]）最简单的解法是什么？有什么缺点？  
（最简单：用全库embedding的均值；缺点：全库均值接近零向量（高维空间均匀分布），归一化后方向随机，推荐结果接近随机。更好方案：热门榜单或请用户选择几个喜欢的类型）

**问 5**：Precision@5=0.6 意味着什么？如何提高它？  
（推荐5首中3首真的被喜欢；提高：更好的embedding（更多对比学习数据）/ 加时间权重（近期历史更重要）/ 考虑多样性（避免只推一种风格））

推导完成后运行下方格验证。

In [ ]:
# ✏️ 对答案格
import numpy as np

# 问1：重心概念验证
embs_q1 = np.array([[1.0, 0.0], [0.0, 1.0], [1.0, 1.0]])
centroid = embs_q1.mean(axis=0)
assert np.allclose(centroid, [2/3, 2/3]), f"重心应=[2/3,2/3]，得{centroid}"
print(f"Q1 ✅  3首歌的重心={centroid}（mean-embedding user profile）")

# 问2：归一化验证
norm_val = np.linalg.norm(centroid)
normalized = centroid / norm_val
assert abs(np.linalg.norm(normalized) - 1.0) < 1e-9, "归一化后模长应=1"
expected = np.array([1/np.sqrt(2), 1/np.sqrt(2)])
assert np.allclose(normalized, expected, atol=1e-6), f"归一化应={expected}，得{normalized}"
print(f"Q2 ✅  user_emb归一化=[{normalized[0]:.4f},{normalized[1]:.4f}]≈[1/√2,1/√2]")

# 问3：过滤效果验证
try:
    np.random.seed(42)
    N_q3, d_q3 = 20, 4
    all_e = np.random.randn(N_q3, d_q3)
    all_e /= np.linalg.norm(all_e, axis=1, keepdims=True)
    history_q3 = [0, 1, 2]
    recs = recommend(history_q3, all_e, top_k=5)
    assert not any(r in history_q3 for r in recs), f"推荐结果{recs}包含历史歌曲{history_q3}"
    print(f"Q3 ✅  推荐结果{recs.tolist()}不含历史{history_q3}（过滤成功）")
except (NotImplementedError, TypeError):
    print(f"Q3 ✅  历史歌曲在user_emb计算中被放大→top位被占→必须过滤（待实现后验证）")

# 问4：冷启动
np.random.seed(7)
N_q4, d_q4 = 100, 8
lib = np.random.randn(N_q4, d_q4)
lib /= np.linalg.norm(lib, axis=1, keepdims=True)
global_mean = lib.mean(axis=0)
global_mean_norm = np.linalg.norm(global_mean)
print(f"Q4 ✅  全库均值模长={global_mean_norm:.4f}（远小于1，高维均匀分布→重心接近零）")

# 问5：Precision@5 计算示例
recommended = np.array([3, 7, 12, 5, 9])
ground_truth = {7, 12, 15, 22, 3}  # 用户后来喜欢的歌
hits = sum(1 for r in recommended if r in ground_truth)
precision_at_5 = hits / len(recommended)
assert abs(precision_at_5 - 0.6) < 1e-9, f"Precision@5应=0.6，得{precision_at_5}"
print(f"Q5 ✅  Precision@5={precision_at_5:.1f}（推荐{list(recommended)[:3]}...中{hits}首被喜欢）")
print("\n🎉 推荐系统白板挑战通过！")

In [ ]:
# ✏️ 本课自评
l81_review = {
    "mean_emb_profile_geometry": None,  # 理解user_emb=历史歌曲均值→语义重心→找最近邻？True/False
    "cold_start_handling":       None,  # 理解冷启动：history=[]时用全库均值或热门榜单？True/False
    "recommend_impl":            None,  # recommend()实现正确：空历史不报错/过滤历史/top-k有序？True/False
    "precision_at_k":            None,  # 理解Precision@k评估推荐质量？True/False
    "whiteboard_passed":         None,  # 白板挑战5道手推全部完成？True/False
}

unfilled = [k for k, v in l81_review.items() if v is None]
assert not unfilled, f'还未填写：{unfilled}'
weak = [k for k, v in l81_review.items() if v is False]
if weak:
    print(f'⚠️  需要加强：{weak}')
else:
    print('✅ L81 全部通关！进入 L82：音乐特征可视化')

## 6. 【可选附录】从推荐到生成：Suno 类音乐 APP 的架构路径

> **注**：本节为扩展阅读，不属于本课核心考核范围。主题聚焦在推荐系统（检索）；生成系统（MusicGen / Suno）将在专项章节深入覆盖。下方 HuggingFace 模板仅作架构参考，不要求执行。

本节做的是**检索**（找相似歌曲）；Suno / MusicGen 做的是**生成**（从文字直接合成音频）。两者核心技术栈高度重叠：

```
检索（本节）:  文字/歌曲 → Embedding → k-NN   → 已有歌曲
生成（Suno）:  文字      → Embedding → LM 生成 → 音频 Token → 音频
```

### 关键组件：EnCodec（音频的 BPE）

Meta **EnCodec** 把连续音频转为离散 token，类比 NLP 的 BPE 词元化：
- 32kHz 音频 → **50 帧/秒 × 4 码书**（musicgen-small）→ 整数序列  
- 压缩比约 **160×**（32000 采样点 / 200 token，即 32000 / (50帧 × 4码书)）；EnCodec Decoder 可无损还原  
- 有了 token，"生成音乐" ≡ "GPT 预测下一个 token"（以文字 embedding 为条件）

### 架构对比

| 系统 | 核心 | 文字接入 | 音频表示 |
|------|------|---------|---------|
| 本节 `recommend()` | 余弦 k-NN | — | 128 维向量 |
| CLAP (LAION) | NT-Xent 对比 | BERT/RoBERTa | 512 维向量 |
| MusicGen (Meta) | Transformer LM | T5 encoder | EnCodec token |
| Suno v3 | Diffusion + LM | 大 LM | 专有编解码器 |

### HuggingFace 一键使用 MusicGen

```python
from transformers import MusicgenForConditionalGeneration, AutoProcessor
model = MusicgenForConditionalGeneration.from_pretrained("facebook/musicgen-small")
processor = AutoProcessor.from_pretrained("facebook/musicgen-small")
inputs = processor(text=["lo-fi hip hop chill beats"], return_tensors="pt")
audio = model.generate(**inputs, max_new_tokens=256)  # ≈5s @50tok/s
```

下方代码展示 EnCodec 压缩原理 + 完整 MusicGen 调用模板（可在有 GPU 的机器上运行）。

In [ ]:
import numpy as np

# ── EnCodec 压缩原理演示（纯公式，无需 GPU / 安装模型）─────────────────────────
sr_audio   = 32_000   # MusicGen 使用 32kHz
duration_s = 1.0
n_samples  = int(sr_audio * duration_s)

frame_rate   = 50    # EnCodec 输出帧率
n_codebooks  = 4     # musicgen-small 用 4 个码书（large 用 8 个）
codebook_sz  = 2048  # 每个码书的词汇量（11 bits）

n_frames     = int(frame_rate * duration_s)
total_tokens = n_frames * n_codebooks
compression  = n_samples / total_tokens

print("=== EnCodec 压缩参数（musicgen-small）===")
print(f"原始音频 : {sr_audio}Hz × {duration_s}s = {n_samples:,} 采样点")
print(f"Token 序列: {n_frames} 帧 × {n_codebooks} 码书 = {total_tokens} 整数")
print(f"压缩比   : {compression:.1f}×")
print(f"每帧信息 : {n_codebooks} × log₂({codebook_sz}) = {n_codebooks * np.log2(codebook_sz):.0f} bits")

# 模拟 token 矩阵
np.random.seed(42)
fake_tokens = np.random.randint(0, codebook_sz, (n_frames, n_codebooks))
print(f"\n模拟 token 矩阵形状: {fake_tokens.shape}  (帧数 × 码书数)")
print("前 5 帧样例:\n", fake_tokens[:5])

# ── MusicGen 完整调用模板（可在有 transformers + GPU 的机器运行）──────────────
MUSICGEN_TEMPLATE = '''
# pip install transformers accelerate scipy  （约 300MB + 模型权重约 1.5GB）
from transformers import MusicgenForConditionalGeneration, AutoProcessor
import torch, scipy.io.wavfile, numpy as np

model = MusicgenForConditionalGeneration.from_pretrained(
    "facebook/musicgen-small",
    torch_dtype=torch.float16,   # 省显存；CPU 用 float32
    device_map="auto",
)
processor = AutoProcessor.from_pretrained("facebook/musicgen-small")
model.eval()

prompts = [
    "lo-fi hip hop, chill, 80 BPM, vinyl crackle",
    "epic orchestral strings and brass, dramatic, 140 BPM",
    "upbeat jazz piano trio, swing, 120 BPM",
]
inputs = processor(text=prompts, padding=True, return_tensors="pt")
inputs = {k: v.to(model.device) for k, v in inputs.items()}

with torch.no_grad():
    # max_new_tokens=256 ≈ 5s @50 tokens/s
    tokens = model.generate(**inputs, max_new_tokens=256,
                             do_sample=True, temperature=1.0, guidance_scale=3.0)

sr = model.config.audio_encoder.sampling_rate  # 32000 Hz
for i, (prompt, audio) in enumerate(zip(prompts, tokens)):
    wav = audio[0].cpu().float().numpy()
    scipy.io.wavfile.write(f"musicgen_{i}.wav", rate=sr, data=wav)
    print(f"[{i}] saved musicgen_{i}.wav  ({len(wav)/sr:.1f}s)")
    print(f"     prompt: {prompt}")
'''

print("\n=== MusicGen HuggingFace 调用模板 ===")
print(MUSICGEN_TEMPLATE)

# ── CLAP 文字-音频对比 embedding 模板 ──────────────────────────────────────────
CLAP_TEMPLATE = '''
# pip install transformers torch torchaudio
from transformers import ClapModel, ClapProcessor
import torch, torchaudio

model     = ClapModel.from_pretrained("laion/larger_clap_general").eval()
processor = ClapProcessor.from_pretrained("laion/larger_clap_general")

# 文字 → embedding
texts  = ["jazz piano", "heavy metal guitar", "classical violin"]
t_inp  = processor(text=texts, return_tensors="pt", padding=True)
with torch.no_grad():
    text_emb = model.get_text_features(**t_inp)      # (3, 512)
    text_emb = text_emb / text_emb.norm(dim=-1, keepdim=True)

# 音频 → embedding（需要真实 WAV，16kHz 单声道）
wav, sr = torchaudio.load("query.wav")
if sr != 48000:
    wav = torchaudio.functional.resample(wav, sr, 48000)
a_inp = processor(audios=wav.squeeze().numpy(), sampling_rate=48000, return_tensors="pt")
with torch.no_grad():
    audio_emb = model.get_audio_features(**a_inp)    # (1, 512)
    audio_emb = audio_emb / audio_emb.norm(dim=-1, keepdim=True)

# 找最匹配的文字标签
sim = (text_emb @ audio_emb.T).squeeze()
print("最匹配:", texts[sim.argmax()])
'''
print("=== CLAP 文字-音频对比 embedding 模板 ===")
print(CLAP_TEMPLATE)

---

→ **下一课**　[L82 · 音乐特征可视化](L82_visual_music.ipynb)

> 下节课将学习 **音乐特征可视化**：色度图、节拍图、相似度热力图动态展示。